# Testando com features selecionadas dado todo o dataset

In [12]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error, r2_score

In [13]:
features_map = {
    "J. Kampe": [
        "z_term_3", "z_term_2", "d_lag_2", "d_lag_3", "d_lag_4", "z_cogram", "d_lag_16", "d_lag_5",
        "d_lag_15", "d_lag_17", "d_lag_12", "d_lag_6", "d_lag_1", "z_term_4", "z_term_6", "d_lag_18",
        "d_lag_13", "d_lag_7", "d_lag_14", "z_cogram_lag_4", "z_gram_lag_4", "z_cogram_lag_5", "z_gram_lag_5", "z_gram_lag_7",
        "z_term_5", "z_gram_lag_6", "z_cogram_lag_6", "z_cogram_lag_7", "d_lag_11", "z_cogram_lag_2", "d_lag_21", "z_cogram_lag_8",
        "z_cogram_lag_11", "z_cogram_lag_3", "d_lag_8", "z_gram_lag_2", "z_gram_lag_8", "z_gram_lag_1", "z_cogram_lag_12", "d_lag_19"
    ]
}

random_forest_features = pd.read_csv("../results/random_forest_feature_selection.csv")["feature"].tolist()
correlation_features = pd.read_csv("../results/correlation_feature_selection.csv")["feature"].tolist()
gevrey_method_features = pd.read_csv("../results/gevrey_method_feature_selection.csv")["feature"].tolist()
mrmr_10_features = pd.read_csv("../results/mrmr_10_features.csv")["feature"].tolist()
mrmr_14_features = pd.read_csv("../results/mrmr_14_features.csv")["feature"].tolist()

features_map["Random Forest (5)"] = random_forest_features[:5]
features_map["Random Forest (10)"] = random_forest_features[:10]
features_map["Random Forest Full"] = random_forest_features
features_map["Correlation"] = correlation_features
features_map["Gevrey Method"] = gevrey_method_features
features_map["Gevrey Method (10 features)"] = gevrey_method_features[:10]  # Limiting to top 10 features
features_map["mRMR (10 features)"] = mrmr_10_features

In [14]:
class DatasetService:
    MAX_LIMIT = 100_000
    OFFSET = 1_000

    def __init__(self, features: list[str]):
        self.X_df = pd.read_csv("../dataset/j_kampe.csv")
        self.y_df = pd.read_csv("../dataset/distances.csv")["distance"]
        self.features = features

    def get_train_test(self, limit: int = 10_000):
        if limit > self.MAX_LIMIT:
            limit = self.MAX_LIMIT

        X = self.X_df[self.features].to_numpy()[self.OFFSET:limit+self.OFFSET]
        y = self.y_df.to_numpy()[self.OFFSET:limit+self.OFFSET]

        split = int(0.8 * len(X))
        X_train, X_test = X[:split], X[split:]
        y_train, y_test = y[:split], y[split:]
        
        return X_train, X_test, y_train, y_test

In [15]:
def run_experiment(features_map):
    results = []

    param_grid = {
        "svr__C": [0.1, 1, 10, 100],
        "svr__epsilon": [0.001, 0.01, 0.1, 0.5],
        "svr__gamma": ["scale", 0.01, 0.1, 1.0]
    }

    for group_name, features in features_map.items():
        data = DatasetService(features)
        X_train, X_test, y_train, y_test = data.get_train_test()

        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("svr", SVR(kernel="rbf"))
        ])

        grid = GridSearchCV(
            pipeline,
            param_grid,
            scoring="neg_root_mean_squared_error",
            cv=TimeSeriesSplit(n_splits=5),
            n_jobs=-1
        )

        grid.fit(X_train, y_train.ravel())

        best_model = grid.best_estimator_

        y_pred = best_model.predict(X_test)

        rmse = root_mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        results.append({
            "Group": group_name,
            "RMSE": rmse,
            "R2": r2,
            "Best C": grid.best_params_["svr__C"],
            "Best epsilon": grid.best_params_["svr__epsilon"],
            "Best gamma": grid.best_params_["svr__gamma"],
            "Features": len(features)
        })

    return pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)


In [16]:
df_standard = run_experiment(features_map)
print(df_standard)
df_standard.to_csv("../results/experiment_7_jkampe.csv", index=False)

                         Group      RMSE        R2  Best C  Best epsilon  \
0                     J. Kampe  0.033892  0.982514      10         0.001   
1                Gevrey Method  0.040265  0.975321      10         0.001   
2           Random Forest (10)  0.040738  0.974737      10         0.001   
3            Random Forest (5)  0.050052  0.961865      10         0.010   
4           Random Forest Full  0.058138  0.948548      10         0.001   
5                  Correlation  0.075211  0.913893      10         0.010   
6           mRMR (10 features)  0.088820  0.879910       1         0.010   
7  Gevrey Method (10 features)  0.096503  0.858238       1         0.010   

  Best gamma  Features  
0       0.01        40  
1       0.01        24  
2      scale        10  
3        1.0         5  
4       0.01        20  
5       0.01        44  
6        0.1        10  
7      scale        10  
